# 🚀 ACE-Net Master Baseline Model Training & Evaluation
### End-to-End Multimodal Deepfake Consistency Training on 14k Preprocessed Dataset

### 🌟 Workflow Overview:
1. **Load Preprocessed Tensors:** Directly streams `.npy` and `.jpg` features from Google Drive.
2. **Load Pretrained Stage-2 Backbone:** Uses `stage2_acenet.pt` from inyong `checkpoints/` folder.
3. **Train on 14,588 Clips (`final_train_manifest.csv`):** Trains with BCE Loss, AdamW, and Cosine Annealing.
4. **Validate Per Epoch (`final_val_manifest.csv`):** Evaluates Accuracy, AUC, and F1 at every epoch and auto-saves the **`best_baseline_model.pth`** to Google Drive!
5. **Final Testing (`final_test_manifest.csv`):** Produces the official Baseline Results Table for your thesis!

## Step 1: Connect to T4 GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name  :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not enabled! Go to Runtime > Change runtime type > T4 GPU!')

## Step 2: Clone Baseline Repository & Checkout Active Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!git log --oneline -1

## Step 3: Install Core Dependencies

In [ ]:
!pip install -q scikit-learn transformers
print('✅ Dependencies installed successfully!')

## Step 3.5: Fast Stage Preprocessed Features to Colab Local NVMe SSD
### ⚡ 20-Second Instant Unzip (kung na-run na ang ZIP notebook) o 64-Thread Turbo Sync!
- Kapag may `baseline_features_all.zip` na sa Drive, **20 seconds lang ang extract**.
- Kung wala pa, awtomatiko nitong gagamitin ang **64 parallel threads** para i-sync.
- Pagkatapos nito, haharurot ang GPU training sa **~0.2s bawat batch** sa Step 4!

In [ ]:
import os, time
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
DRIVE_ZIP = DRIVE_BASE / 'baseline_features_all.zip'
LOCAL_ROOT = Path('/content/preprocessed_local')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

if DRIVE_ZIP.exists():
    print('=' * 75)
    print(f'📦 Found master zip in Google Drive: {DRIVE_ZIP}')
    print(f'   Size: {DRIVE_ZIP.stat().st_size / (1024**3):.2f} GB')
    print('⚡ Instant unzipping directly to Colab Local NVMe SSD...')
    print('=' * 75)
    start_u = time.time()
    !unzip -q -o "{DRIVE_ZIP}" -d "{LOCAL_ROOT}"
    print(f'\n✅ Unzip Complete in {time.time() - start_u:.1f} seconds!')
    print(f'   Ready folders: {[p.name for p in LOCAL_ROOT.iterdir() if p.is_dir()]}')
    print('⚡ GPU is ready for 0.2s/batch ultra-fast training!')
else:
    print('🚀 Master zip not found on Drive. Running 64-thread parallel sync...')
    !python -m scripts.fast_multithread_sync \
        --drive-root "{DRIVE_BASE}/Baseline preprocessed" \
        --local-root "{LOCAL_ROOT}" \
        --workers 64


## Step 4: Run Stage-2 Baseline Training & Evaluation (1-Click Run)